<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Block 1 — Load 19-column final dataset**

In [2]:
# ============================================================
# ML-05 / W05 — BLOCK 1
# LOAD FINAL 19-COLUMN FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "/content/final_features_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 70)
print("FINAL FEATURE DATASET")
print("=" * 70)

print("Rows    :", f"{len(df):,}")
print("Columns :", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

print("\nShape:", df.shape)

FINAL FEATURE DATASET
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape: (2871202, 19)


**Block 2 — Missing indicators + 90-day eligibility**

In [3]:
# ============================================================
# BLOCK 2
# MISSING INDICATORS + BASIC DATA QUALITY
# ============================================================

df["month"] = pd.to_datetime(
    df["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Remove meaningless missing indicator
# ------------------------------------------------------------

DROP_INDICATOR = "ai_other_missing"

if DROP_INDICATOR in df.columns:
    df = df.drop(columns=[DROP_INDICATOR])

print("=" * 70)
print("MISSING INDICATOR DECISION")
print("=" * 70)

print("Dropped:", DROP_INDICATOR)

remaining_indicators = [
    c for c in df.columns
    if c.endswith("_missing")
]

print("\nMissing indicators kept:")
for c in remaining_indicators:
    print("KEEP ->", c)

# ------------------------------------------------------------
# 2. Sort page-wise chronologically
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Check page history
# ------------------------------------------------------------

page_history = (
    df
    .groupby("content_hash_id")["month"]
    .nunique()
)

print("\n" + "=" * 70)
print("PAGE HISTORY")
print("=" * 70)

print(
    "Total pages:",
    f"{len(page_history):,}"
)

print(
    "Pages with < 3 months:",
    f"{(page_history < 3).sum():,}"
)

print(
    "Pages with >= 3 months:",
    f"{(page_history >= 3).sum():,}"
)

# ------------------------------------------------------------
# 4. Check monthly continuity
# ------------------------------------------------------------

df["previous_month"] = (
    df
    .groupby("content_hash_id")["month"]
    .shift(1)
)

df["month_gap"] = (
    (
        df["month"].dt.year
        - df["previous_month"].dt.year
    ) * 12
    +
    (
        df["month"].dt.month
        - df["previous_month"].dt.month
    )
)

gap_rows = df[
    df["month_gap"].notna()
    &
    (df["month_gap"] > 1)
]

print("\n" + "=" * 70)
print("MONTHLY CONTINUITY")
print("=" * 70)

print(
    "Rows with a month gap:",
    f"{len(gap_rows):,}"
)

print(
    "Pages affected:",
    f"{gap_rows['content_hash_id'].nunique():,}"
)

# helper columns no longer needed
df = df.drop(
    columns=[
        "previous_month",
        "month_gap"
    ]
)

MISSING INDICATOR DECISION
Dropped: ai_other_missing

Missing indicators kept:
KEEP -> gsc_avg_position_missing
KEEP -> ga4_total_engagement_sec_missing
KEEP -> sessions_organic_missing
KEEP -> sessions_ai_missing

PAGE HISTORY
Total pages: 427,292
Pages with < 3 months: 47,141
Pages with >= 3 months: 380,151

MONTHLY CONTINUITY
Rows with a month gap: 13,014
Pages affected: 11,525


**Block 3 — Feature impact + leakage checks**

In [4]:
# ============================================================
# BLOCK 3
# FEATURE AUDIT + LEAKAGE + VIF
# ============================================================

from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------------------------------------------------
# 1. ID columns are NOT model features
# ------------------------------------------------------------

ID_COLUMNS = [
    "client_hash_id",
    "content_hash_id"
]

# ------------------------------------------------------------
# 2. Potential future/label-derived names
# ------------------------------------------------------------

future_keywords = [
    "future",
    "target",
    "label",
    "decay",
    "next_",
    "lead_"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in future_keywords
    ):
        suspicious_columns.append(col)

print("=" * 70)
print("LEAKAGE NAME CHECK")
print("=" * 70)

if suspicious_columns:
    print("Potentially suspicious columns:")
    for col in suspicious_columns:
        print("CHECK ->", col)
else:
    print("PASS: No obvious future/target-derived columns.")

# ------------------------------------------------------------
# 3. Constant columns
# ------------------------------------------------------------

feature_candidates = [
    c for c in df.columns
    if c not in ID_COLUMNS + ["month"]
]

constant_columns = [
    c for c in feature_candidates
    if df[c].nunique(dropna=False) <= 1
]

print("\n" + "=" * 70)
print("ZERO-VARIANCE CHECK")
print("=" * 70)

if constant_columns:
    for c in constant_columns:
        print("DROP ->", c)
else:
    print("PASS: No zero-variance columns.")

# ------------------------------------------------------------
# 4. Numeric feature list
# ------------------------------------------------------------

numeric_features = [
    c for c in feature_candidates
    if c not in constant_columns
    and pd.api.types.is_numeric_dtype(df[c])
]

print("\nNumeric model candidates:")
for c in numeric_features:
    print(" -", c)

# ------------------------------------------------------------
# 5. Correlation between current features
# ------------------------------------------------------------

corr_matrix = (
    df[numeric_features]
    .corr()
)

print("\n" + "=" * 70)
print("HIGH FEATURE-TO-FEATURE CORRELATION")
print("=" * 70)

high_corr_pairs = []

for i in range(len(numeric_features)):

    for j in range(i + 1, len(numeric_features)):

        a = numeric_features[i]
        b = numeric_features[j]

        corr = corr_matrix.loc[a, b]

        if abs(corr) >= 0.90:

            high_corr_pairs.append(
                (a, b, round(corr, 3))
            )

if high_corr_pairs:

    for pair in high_corr_pairs:
        print(pair)

else:

    print("No feature pairs with |correlation| >= 0.90")

# ------------------------------------------------------------
# 6. VIF sample
# ------------------------------------------------------------
# VIF on millions of rows is unnecessarily expensive.
# We use a reproducible sample for diagnostic purposes.

vif_sample = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .sample(
        n=min(100_000, len(df)),
        random_state=42
    )
)

# Remove columns with zero variance inside sample
vif_features = [
    c for c in numeric_features
    if vif_sample[c].nunique() > 1
]

X_vif = vif_sample[vif_features]

vif_table = pd.DataFrame({
    "feature": vif_features,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            i
        )
        for i in range(X_vif.shape[1])
    ]
})

vif_table = (
    vif_table
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("VIF TEST")
print("=" * 70)

display(vif_table)

LEAKAGE NAME CHECK
PASS: No obvious future/target-derived columns.

ZERO-VARIANCE CHECK
PASS: No zero-variance columns.

Numeric model candidates:
 - gsc_clicks
 - gsc_impressions
 - gsc_avg_position
 - ga4_total_engagement_sec
 - sessions_organic
 - sessions_ai
 - missing_count
 - gsc_avg_position_missing
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing
 - ctr
 - sec_per_click
 - ai_share
 - engagement_per_organic_session

HIGH FEATURE-TO-FEATURE CORRELATION
('gsc_clicks', 'sessions_organic', np.float64(0.978))
('missing_count', 'ga4_total_engagement_sec_missing', np.float64(0.968))
('missing_count', 'sessions_organic_missing', np.float64(0.968))
('missing_count', 'sessions_ai_missing', np.float64(0.968))
('ga4_total_engagement_sec_missing', 'sessions_organic_missing', np.float64(1.0))
('ga4_total_engagement_sec_missing', 'sessions_ai_missing', np.float64(1.0))
('sessions_organic_missing', 'sessions_ai_missing', np.float64(1.0))


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIF TEST


,feature,VIF
0,missing_count,inf
1,ga4_total_engagement_sec_missing,inf
2,gsc_avg_position_missing,inf
3,sessions_ai_missing,inf
4,sessions_organic_missing,inf
5,sessions_organic,4.518367
6,gsc_clicks,3.863913
7,gsc_impressions,2.184852
8,ga4_total_engagement_sec,2.176906
9,sessions_ai,1.924955


**Block 3.5: keep selected impactful  features after VIF and correlation test**

In [5]:
# ============================================================
# ML-05 → ML-06/07 PREPARATION
# FINAL FEATURE SET + 90-DAY ELIGIBILITY CHECK
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("FINAL FEATURE DATASET PREPARATION")
print("=" * 70)

# ============================================================
# 1. FIND THE CURRENT 19-COLUMN DATAFRAME
# ============================================================

# Try known dataframe names first
candidate_names = [
    "df_baseline",
    "df_model_base",
    "df_features_clean",
    "df_clean",
    "df"
]

source_df = None
source_name = None

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            source_df = obj.copy()
            source_name = name
            break

if source_df is None:
    raise NameError(
        "No suitable dataframe found. "
        "Please load your 19-column Parquet dataset first."
    )

print(f"Source dataframe used: {source_name}")
print(f"Rows before cleaning: {len(source_df):,}")
print(f"Columns before cleaning: {source_df.shape[1]}")

# ============================================================
# 2. START FROM SOURCE DATA
# ============================================================

df_model_base = source_df.copy()

# Convert month safely
df_model_base["month"] = pd.to_datetime(
    df_model_base["month"],
    errors="coerce"
)

# ============================================================
# 3. REMOVE REDUNDANT MISSINGNESS FEATURES
# ============================================================

remove_columns = [
    "missing_count",
    "ga4_total_engagement_sec_missing",
    "sessions_organic_missing",
    "sessions_ai_missing"
]

# Only remove columns that actually exist
remove_columns = [
    col
    for col in remove_columns
    if col in df_model_base.columns
]

df_model_base = df_model_base.drop(
    columns=remove_columns
)

print("\n" + "=" * 70)
print("REMOVED REDUNDANT FEATURES")
print("=" * 70)

if remove_columns:
    for col in remove_columns:
        print(" -", col)
else:
    print("None")

# ============================================================
# 4. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "content_hash_id",
    "month"
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model_base.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

# ============================================================
# 5. REMOVE INVALID MONTH ROWS
# ============================================================

before_month_filter = len(df_model_base)

df_model_base = df_model_base[
    df_model_base["month"].notna()
].copy()

removed_invalid_months = (
    before_month_filter -
    len(df_model_base)
)

print(
    f"\nRows removed because month was invalid: "
    f"{removed_invalid_months:,}"
)

# ============================================================
# 6. REMOVE DUPLICATE PAGE-MONTH RECORDS IF ANY
# ============================================================

duplicate_count = (
    df_model_base
    .duplicated(
        subset=["content_hash_id", "month"]
    )
    .sum()
)

print(
    f"Duplicate page-month rows found: "
    f"{duplicate_count:,}"
)

if duplicate_count > 0:
    df_model_base = (
        df_model_base
        .drop_duplicates(
            subset=["content_hash_id", "month"],
            keep="first"
        )
        .copy()
    )

# ============================================================
# 7. SORT PAGE HISTORIES CHRONOLOGICALLY
# ============================================================

df_model_base = (
    df_model_base
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 8. CALCULATE OBSERVED HISTORY
# ============================================================

first_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_model_base
    .groupby("content_hash_id")["month"]
    .transform("max")
)

df_model_base["content_age_days"] = (
    last_month - first_month
).dt.days

# ============================================================
# 9. 90-DAY ELIGIBILITY
# ============================================================

eligible_mask = (
    df_model_base["content_age_days"] >= 90
)

df_model_eligible = (
    df_model_base.loc[eligible_mask]
    .copy()
)

removed_rows = (
    len(df_model_base) -
    len(df_model_eligible)
)

print("\n" + "=" * 70)
print("90-DAY ELIGIBILITY")
print("=" * 70)

print(
    f"Rows before 90-day filter : "
    f"{len(df_model_base):,}"
)

print(
    f"Rows after 90-day filter  : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Rows removed              : "
    f"{removed_rows:,}"
)

# ============================================================
# 10. PAGE-LEVEL ELIGIBILITY CHECK
# ============================================================

page_age = (
    df_model_eligible
    .groupby("content_hash_id")["content_age_days"]
    .max()
)

if len(page_age) > 0:

    print(
        f"\nEligible pages: "
        f"{len(page_age):,}"
    )

    print(
        f"Minimum eligible history: "
        f"{page_age.min()} days"
    )

    print(
        f"Pages with <90 days remaining: "
        f"{(page_age < 90).sum():,}"
    )

    # Safety check
    assert (
        page_age >= 90
    ).all(), (
        "ERROR: A page below 90 days remains."
    )

else:
    raise ValueError(
        "No pages remain after the 90-day eligibility filter."
    )

# ============================================================
# 11. FINAL CHRONOLOGICAL ORDER
# ============================================================

df_model_eligible = (
    df_model_eligible
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

# ============================================================
# 12. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Final rows    : "
    f"{len(df_model_eligible):,}"
)

print(
    f"Final columns : "
    f"{df_model_eligible.shape[1]}"
)

print(
    f"Final shape   : "
    f"{df_model_eligible.shape}"
)

print("\nRemaining columns:")

for i, col in enumerate(
    df_model_eligible.columns,
    start=1
):
    print(f"{i:2}. {col}")

# ============================================================
# 13. FINAL 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET — FIRST 5 ROWS")
print("=" * 70)

display(
    df_model_eligible.head(5)
)

# ============================================================
# 14. FINAL 90-DAY SAFETY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SAFETY CHECK")
print("=" * 70)

print(
    "Minimum content_age_days:",
    df_model_eligible["content_age_days"].min()
)

print(
    "Rows below 90 days:",
    (
        df_model_eligible["content_age_days"] < 90
    ).sum()
)

assert (
    df_model_eligible["content_age_days"] >= 90
).all()

print("PASS: No page below the 90-day requirement.")

# ============================================================
# 15. SAVE FINAL INTERMEDIATE DATASET
# ============================================================

output_path = (
    "/content/final_model_eligible.parquet"
)

df_model_eligible.to_parquet(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("PARQUET SAVED")
print("=" * 70)

print(output_path)

print("\n" + "=" * 70)
print("PREPARATION COMPLETE")
print("=" * 70)

FINAL FEATURE DATASET PREPARATION
Source dataframe used: df
Rows before cleaning: 2,871,202
Columns before cleaning: 18

REMOVED REDUNDANT FEATURES
 - missing_count
 - ga4_total_engagement_sec_missing
 - sessions_organic_missing
 - sessions_ai_missing

Rows removed because month was invalid: 0
Duplicate page-month rows found: 0

90-DAY ELIGIBILITY
Rows before 90-day filter : 2,871,202
Rows after 90-day filter  : 2,705,303
Rows removed              : 165,899

Eligible pages: 349,557
Minimum eligible history: 92 days
Pages with <90 days remaining: 0

FINAL DATASET
Final rows    : 2,705,303
Final columns : 15
Final shape   : (2705303, 15)

Remaining columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. gsc_avg_position_missing
11. ctr
12. sec_per_click
13. ai_share
14. engagement_per_organic_session
15. content_age_days

FINAL DATASET — FIRST 5 ROWS


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,gsc_avg_position_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session,content_age_days
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0.006849,0.0,0.0,0.0,457
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0.000000,0.0,0.0,0.0,457



FINAL SAFETY CHECK
Minimum content_age_days: 92
Rows below 90 days: 0
PASS: No page below the 90-day requirement.

PARQUET SAVED
/content/final_model_eligible.parquet

PREPARATION COMPLETE


**Block 4 — Rolling 90-day features + target**

In [6]:
# ============================================================
# BLOCK 4 — FAST ROLLING 3-MONTH WINDOW
# CURRENT 3 MONTHS → NEXT 3 MONTHS
# TARGET = DOWN / FLAT / UP
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("BLOCK 4 — FAST ROLLING WINDOW + TARGET CREATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Start from final eligible dataset
# ------------------------------------------------------------

df_roll = df_model_eligible.copy()

df_roll["month"] = pd.to_datetime(
    df_roll["month"],
    errors="coerce"
)

df_roll = df_roll.dropna(
    subset=["content_hash_id", "month"]
)

# Sort once
df_roll = df_roll.sort_values(
    ["content_hash_id", "month"]
).reset_index(drop=True)

print(f"Input rows: {len(df_roll):,}")

# ------------------------------------------------------------
# 2. Check duplicate page-month
# ------------------------------------------------------------

duplicates = df_roll.duplicated(
    ["content_hash_id", "month"]
).sum()

print(f"Duplicate page-month rows: {duplicates:,}")

if duplicates > 0:
    raise ValueError(
        "Duplicate page-month rows found. "
        "Resolve duplicates before rolling-window creation."
    )

# ------------------------------------------------------------
# 3. Convert month to integer month index
# ------------------------------------------------------------

df_roll["_month_idx"] = (
    df_roll["month"].dt.year * 12
    + df_roll["month"].dt.month
)

# ------------------------------------------------------------
# 4. Identify consecutive monthly observations
# ------------------------------------------------------------

g = df_roll.groupby(
    "content_hash_id",
    sort=False
)

df_roll["_prev_month_idx"] = g["_month_idx"].shift(1)

df_roll["_gap"] = (
    df_roll["_month_idx"]
    - df_roll["_prev_month_idx"]
)

# A new sequence starts when:
# - first observation
# - OR a month is missing

df_roll["_new_sequence"] = (
    df_roll["_prev_month_idx"].isna()
    | (df_roll["_gap"] != 1)
)

df_roll["_sequence_id"] = (
    df_roll["_new_sequence"]
    .groupby(df_roll["content_hash_id"])
    .cumsum()
)

# ------------------------------------------------------------
# 5. Create row number inside each continuous sequence
# ------------------------------------------------------------

df_roll["_pos"] = (
    df_roll
    .groupby(
        ["content_hash_id", "_sequence_id"],
        sort=False
    )
    .cumcount()
)

# ------------------------------------------------------------
# 6. Fast rolling values
# ------------------------------------------------------------

group_cols = [
    "content_hash_id",
    "_sequence_id"
]

# Current 3-month impression average
df_roll["_current_imp_3m"] = (
    df_roll
    .groupby(group_cols)["gsc_impressions"]
    .transform(
        lambda x: x.rolling(3).mean()
    )
)

# Future 3-month impression average:
# shift(-3) means look 3 months ahead,
# then rolling 3 observations forward.
df_roll["_future_imp_3m"] = (
    df_roll
    .groupby(group_cols)["gsc_impressions"]
    .transform(
        lambda x: x.shift(-3).rolling(3).mean()
    )
)

# ------------------------------------------------------------
# IMPORTANT:
# The rolling calculation above is mathematically correct,
# but we need a clean window-start representation.
#
# We therefore calculate using positional shifts.
# ------------------------------------------------------------

g2 = df_roll.groupby(
    group_cols,
    sort=False
)

# Current window:
df_roll["current_imp_3m"] = (
    df_roll["gsc_impressions"]
    + g2["gsc_impressions"].shift(1)
    + g2["gsc_impressions"].shift(2)
) / 3

# Future window:
df_roll["future_imp_3m"] = (
    g2["gsc_impressions"].shift(-3)
    + g2["gsc_impressions"].shift(-4)
    + g2["gsc_impressions"].shift(-5)
) / 3

# ------------------------------------------------------------
# 7. Verify six consecutive observations
# ------------------------------------------------------------

# For a valid window starting at row i:
# i, i+1, i+2 = current 3 months
# i+3, i+4, i+5 = future 3 months

pos = df_roll["_pos"]

df_roll["_valid_window"] = (
    df_roll["current_imp_3m"].notna()
    & df_roll["future_imp_3m"].notna()
)

# ------------------------------------------------------------
# 8. Current month / future month information
# ------------------------------------------------------------

df_roll["window_start"] = df_roll["month"]

df_roll["window_end"] = (
    g2["month"].shift(-2)
)

df_roll["future_start"] = (
    g2["month"].shift(-3)
)

df_roll["future_end"] = (
    g2["month"].shift(-5)
)

# ------------------------------------------------------------
# 9. Calculate future impression change
# ------------------------------------------------------------

df_roll["future_impression_change_pct"] = np.where(
    (
        df_roll["_valid_window"]
        & (df_roll["current_imp_3m"] > 0)
    ),
    (
        (
            df_roll["future_imp_3m"]
            - df_roll["current_imp_3m"]
        )
        / df_roll["current_imp_3m"]
    ) * 100,
    np.nan
)

# ------------------------------------------------------------
# 10. Keep only valid 6-month windows
# ------------------------------------------------------------

df_windows = df_roll.loc[
    df_roll["_valid_window"]
].copy()

# ------------------------------------------------------------
# 11. Create target
# ------------------------------------------------------------

df_windows["target"] = np.select(
    [
        df_windows["future_impression_change_pct"] <= -20,
        df_windows["future_impression_change_pct"] >= 20
    ],
    [
        "Down",
        "Up"
    ],
    default="Flat"
)

# ------------------------------------------------------------
# 12. Current-window features
# ------------------------------------------------------------

numeric_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "gsc_avg_position_missing",
    "ai_other_missing",
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

# Create 3-month mean for each feature
# using already sorted continuous sequences.

for feature in numeric_features:

    if feature not in df_windows.columns:
        continue

    shifted_1 = g2[feature].shift(1)
    shifted_2 = g2[feature].shift(2)

    df_windows[f"{feature}_mean_3m"] = (
        df_windows[feature]
        + shifted_1.loc[df_windows.index]
        + shifted_2.loc[df_windows.index]
    ) / 3

# ------------------------------------------------------------
# 13. Keep clean columns only
# ------------------------------------------------------------

keep_columns = [
    "content_hash_id",
    "client_hash_id",
    "window_start",
    "window_end",
    "future_start",
    "future_end",
    "current_imp_3m",
    "future_imp_3m",
    "future_impression_change_pct",
    "target"
]

# Add engineered current-window features
feature_columns = [
    col for col in df_windows.columns
    if col.endswith("_mean_3m")
]

keep_columns.extend(feature_columns)

df_windows = (
    df_windows[keep_columns]
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 14. Target distribution
# ------------------------------------------------------------

target_summary = (
    df_windows["target"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="observations")
)

target_summary["percentage"] = (
    target_summary["observations"]
    / len(df_windows)
    * 100
).round(2)

# ------------------------------------------------------------
# 15. Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ROLLING WINDOW RESULT")
print("=" * 70)

print(
    f"Valid rolling training rows : "
    f"{len(df_windows):,}"
)

print(
    f"Unique pages                 : "
    f"{df_windows['content_hash_id'].nunique():,}"
)

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

display(target_summary)

print("\n" + "=" * 70)
print("WINDOW PREVIEW")
print("=" * 70)

display(
    df_windows[
        [
            "content_hash_id",
            "window_start",
            "window_end",
            "future_start",
            "future_end",
            "current_imp_3m",
            "future_imp_3m",
            "future_impression_change_pct",
            "target"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 16. Leakage check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE SAFETY CHECK")
print("=" * 70)

future_cols = [
    col for col in df_windows.columns
    if (
        "future_" in col
        or "change_pct" in col
        or col == "target"
    )
]

print("Future/target columns:")
for col in future_cols:
    print(" -", col)

print(
    "\nThese columns are target-generation columns "
    "and must NOT be used as model inputs."
)

# ------------------------------------------------------------
# 17. Save
# ------------------------------------------------------------

output_path = "/content/rolling_3m_training_dataset.parquet"

df_windows.to_parquet(
    output_path,
    index=False
)

print("\nSaved:")
print(output_path)

print("\n" + "=" * 70)
print("BLOCK 4 COMPLETE")
print("=" * 70)

BLOCK 4 — FAST ROLLING WINDOW + TARGET CREATION
Input rows: 2,705,303
Duplicate page-month rows: 0

ROLLING WINDOW RESULT
Valid rolling training rows : 497,837
Unique pages                 : 232,933

TARGET DISTRIBUTION


,target,observations,percentage
0,Up,198581,39.89
1,Flat,176800,35.51
2,Down,122456,24.60



WINDOW PREVIEW


,content_hash_id,window_start,window_end,future_start,future_end,current_imp_3m,future_imp_3m,future_impression_change_pct,target
0,content_000005d4ced12088,2025-05-01,2025-07-01,2025-08-01,2025-10-01,136.666667,487.666667,256.829268,Up
1,content_000005d4ced12088,2025-06-01,2025-08-01,2025-09-01,2025-11-01,180.666667,280.333333,55.166052,Up
2,content_000005d4ced12088,2025-07-01,2025-09-01,2025-10-01,2025-12-01,216.666667,167.000000,-22.923077,Down
3,content_000005d4ced12088,2025-08-01,2025-10-01,2025-11-01,2026-01-01,378.333333,106.333333,-71.894273,Down
4,content_000005d4ced12088,2025-09-01,2025-11-01,2025-12-01,2026-02-01,506.666667,74.333333,-85.328947,Down
5,content_000005d4ced12088,2025-10-01,2025-12-01,2026-01-01,2026-03-01,487.666667,41.666667,-91.455913,Down
6,content_000005d4ced12088,2025-11-01,2026-01-01,2026-02-01,2026-04-01,280.333333,63.666667,-77.288942,Down
7,content_000005d4ced12088,2025-12-01,2026-02-01,2026-03-01,2026-05-01,167.000000,82.666667,-50.499002,Down
8,content_000005d4ced12088,2026-01-01,2026-03-01,2026-04-01,2026-06-01,106.333333,78.333333,-26.332288,Down
9,content_00001e488b74b799,2026-01-01,2026-03-01,2026-04-01,2026-06-01,0.000000,0.000000,NaN,Flat



LEAKAGE SAFETY CHECK
Future/target columns:
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct
 - target

These columns are target-generation columns and must NOT be used as model inputs.

Saved:
/content/rolling_3m_training_dataset.parquet

BLOCK 4 COMPLETE


**Add remaining Features with rolling window table**

In [12]:
# ============================================================
# BLOCK 4 — FAST ROLLING 3M → FUTURE 3M WINDOWS
# Current 3 months = FEATURES
# Next 3 months = TARGET GENERATION ONLY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("BLOCK 4 — FAST ROLLING WINDOW + TARGET CREATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. COPY FINAL ELIGIBLE DATASET
# ------------------------------------------------------------

df_roll = df_model_eligible.copy()

df_roll["month"] = pd.to_datetime(
    df_roll["month"],
    errors="coerce"
)

df_roll = df_roll.dropna(
    subset=["content_hash_id", "month"]
).copy()

# Remove duplicate page-month rows safely
before_dup = len(df_roll)

df_roll = (
    df_roll
    .drop_duplicates(
        subset=["content_hash_id", "month"],
        keep="first"
    )
    .sort_values(
        ["content_hash_id", "month"]
    )
    .reset_index(drop=True)
)

print(f"Rows after duplicate cleanup: {len(df_roll):,}")
print(f"Duplicate rows removed: {before_dup - len(df_roll):,}")

# ------------------------------------------------------------
# 2. MONTH PERIOD
# ------------------------------------------------------------

df_roll["month_period"] = df_roll["month"].dt.to_period("M")

# ------------------------------------------------------------
# 3. FAST CONSECUTIVE 6-MONTH CHECK
#
# Row i must have:
# i+1 = next month
# i+2 = next month
# ...
# i+5 = next month
# ------------------------------------------------------------

page = df_roll["content_hash_id"]
period = df_roll["month_period"]

valid_6m = (
    page.eq(page.shift(-1))
    & page.eq(page.shift(-2))
    & page.eq(page.shift(-3))
    & page.eq(page.shift(-4))
    & page.eq(page.shift(-5))
    & period.add(1).eq(period.shift(-1))
    & period.add(2).eq(period.shift(-2))
    & period.add(3).eq(period.shift(-3))
    & period.add(4).eq(period.shift(-4))
    & period.add(5).eq(period.shift(-5))
)

valid_idx = np.flatnonzero(valid_6m.to_numpy())

print("\n" + "=" * 70)
print("VALID 6-MONTH WINDOWS")
print("=" * 70)

print(f"Valid current→future windows: {len(valid_idx):,}")

# ------------------------------------------------------------
# 4. FEATURES FROM CURRENT 3 MONTHS
# ------------------------------------------------------------

feature_columns = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "gsc_avg_position_missing",
    "ai_other_missing",
    "ctr",
    "sec_per_click",
    "ai_share",
    "engagement_per_organic_session"
]

# Keep only features that actually exist
feature_columns = [
    c for c in feature_columns
    if c in df_roll.columns
]

print("\nCurrent-window feature columns:")

for i, col in enumerate(feature_columns, 1):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# 5. CREATE OUTPUT USING NUMPY ARRAYS
# ------------------------------------------------------------

out = pd.DataFrame(index=np.arange(len(valid_idx)))

# Basic identifiers
out["content_hash_id"] = (
    df_roll["content_hash_id"].iloc[valid_idx].to_numpy()
)

if "client_hash_id" in df_roll.columns:
    out["client_hash_id"] = (
        df_roll["client_hash_id"].iloc[valid_idx].to_numpy()
    )

# Current window dates
out["window_start"] = (
    df_roll["month"].iloc[valid_idx].to_numpy()
)

out["window_end"] = (
    df_roll["month"].iloc[valid_idx + 2].to_numpy()
)

# Future window dates
out["future_start"] = (
    df_roll["month"].iloc[valid_idx + 3].to_numpy()
)

out["future_end"] = (
    df_roll["month"].iloc[valid_idx + 5].to_numpy()
)

# ------------------------------------------------------------
# 6. CURRENT 3-MONTH FEATURES
#
# mean_3m = average of current 3 months
# last = latest month of current window
# ------------------------------------------------------------

for col in feature_columns:

    values = pd.to_numeric(
        df_roll[col],
        errors="coerce"
    ).to_numpy()

    v0 = values[valid_idx]
    v1 = values[valid_idx + 1]
    v2 = values[valid_idx + 2]

    out[f"{col}_mean_3m"] = (
        (v0 + v1 + v2) / 3
    )

    out[f"{col}_last"] = v2

# ------------------------------------------------------------
# 7. CURRENT IMPRESSIONS
# ------------------------------------------------------------

imp = pd.to_numeric(
    df_roll["gsc_impressions"],
    errors="coerce"
).to_numpy()

current_imp_3m = (
    imp[valid_idx]
    + imp[valid_idx + 1]
    + imp[valid_idx + 2]
) / 3

future_imp_3m = (
    imp[valid_idx + 3]
    + imp[valid_idx + 4]
    + imp[valid_idx + 5]
) / 3

out["current_imp_3m"] = current_imp_3m
out["future_imp_3m"] = future_imp_3m

# ------------------------------------------------------------
# 8. FUTURE IMPRESSION CHANGE
# ------------------------------------------------------------

valid_change = (
    np.isfinite(current_imp_3m)
    & np.isfinite(future_imp_3m)
    & (current_imp_3m > 0)
)

out["future_impression_change_pct"] = np.nan

out.loc[valid_change, "future_impression_change_pct"] = (
    (
        (
            future_imp_3m[valid_change]
            - current_imp_3m[valid_change]
        )
        / current_imp_3m[valid_change]
    ) * 100
)

# Remove invalid target-generation rows
out = out[
    out["future_impression_change_pct"].notna()
].copy()

# ------------------------------------------------------------
# 9. TARGET
#
# <= -20%  -> Down
# >= +20%  -> Up
# otherwise -> Flat
#
# This is our transparent working threshold.
# ------------------------------------------------------------

out["target"] = np.select(
    [
        out["future_impression_change_pct"] <= -20,
        out["future_impression_change_pct"] >= 20
    ],
    [
        "Down",
        "Up"
    ],
    default="Flat"
)

# ------------------------------------------------------------
# 10. TARGET DISTRIBUTION
# ------------------------------------------------------------

target_summary = (
    out["target"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="observations")
)

target_summary["percentage"] = (
    target_summary["observations"]
    / len(out)
    * 100
).round(2)

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

display(target_summary)

# ------------------------------------------------------------
# 11. LEAKAGE COLUMNS
# ------------------------------------------------------------

target_generation_columns = [
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct",
    "target"
]

print("\n" + "=" * 70)
print("TARGET-GENERATION COLUMNS")
print("=" * 70)

for col in target_generation_columns:
    print(" -", col)

print(
    "\nThese columns are NOT model input features."
)

# ------------------------------------------------------------
# 12. FINAL MODEL INPUT COLUMNS
# ------------------------------------------------------------

model_exclude = (
    [
        "content_hash_id",
        "client_hash_id",
        "window_start",
        "window_end",
    ]
    + target_generation_columns
)

model_feature_columns = [
    col
    for col in out.columns
    if col not in model_exclude
]

print("\n" + "=" * 70)
print("FINAL MODEL INPUT FEATURES")
print("=" * 70)

for i, col in enumerate(model_feature_columns, 1):
    print(f"{i:2}. {col}")

# ------------------------------------------------------------
# 13. FINAL SHAPE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL WINDOW DATASET")
print("=" * 70)

print(f"Rows remaining : {len(out):,}")
print(f"Columns        : {out.shape[1]}")
print(
    f"Model features : {len(model_feature_columns)}"
)

# ------------------------------------------------------------
# 14. PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 5 ROWS")
print("=" * 70)

display(out.head(5))

# ------------------------------------------------------------
# 15. SAFETY CHECK
# ------------------------------------------------------------

assert (
    out["future_impression_change_pct"].notna().all()
)

assert (
    out["target"].isin(
        ["Down", "Flat", "Up"]
    ).all()
)

print("\n" + "=" * 70)
print("BLOCK 4 COMPLETE")
print("=" * 70)

BLOCK 4 — FAST ROLLING WINDOW + TARGET CREATION
Rows after duplicate cleanup: 2,705,303
Duplicate rows removed: 0

VALID 6-MONTH WINDOWS
Valid current→future windows: 978,801

Current-window feature columns:
 1. gsc_clicks
 2. gsc_impressions
 3. gsc_avg_position
 4. ga4_total_engagement_sec
 5. sessions_organic
 6. sessions_ai
 7. gsc_avg_position_missing
 8. ctr
 9. sec_per_click
10. ai_share
11. engagement_per_organic_session

TARGET DISTRIBUTION


,target,observations,percentage
0,Up,312383,49.83
1,Down,228624,36.47
2,Flat,85829,13.69



TARGET-GENERATION COLUMNS
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct
 - target

These columns are NOT model input features.

FINAL MODEL INPUT FEATURES
 1. gsc_clicks_mean_3m
 2. gsc_clicks_last
 3. gsc_impressions_mean_3m
 4. gsc_impressions_last
 5. gsc_avg_position_mean_3m
 6. gsc_avg_position_last
 7. ga4_total_engagement_sec_mean_3m
 8. ga4_total_engagement_sec_last
 9. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m

FINAL WINDOW DATASET
Rows remaining : 626,836
Columns        : 32
Model features : 23

FIRST 5 ROWS


,content_hash_id,client_hash_id,window_start,window_end,future_start,future_end,gsc_clicks_mean_3m,gsc_clicks_last,gsc_impressions_mean_3m,gsc_impressions_last,...,sec_per_click_mean_3m,sec_per_click_last,ai_share_mean_3m,ai_share_last,engagement_per_organic_session_mean_3m,engagement_per_organic_session_last,current_imp_3m,future_imp_3m,future_impression_change_pct,target
0,content_000005d4ced12088,client_9958f0a7ae1df715,2025-03-01,2025-05-01,2025-06-01,2025-08-01,0.333333,0.0,136.666667,257.0,...,0.0,0.0,0.0,0.0,0.0,0.0,136.666667,378.333333,176.829268,Up
1,content_000005d4ced12088,client_9958f0a7ae1df715,2025-04-01,2025-06-01,2025-07-01,2025-09-01,0.333333,0.0,180.666667,139.0,...,0.0,0.0,0.0,0.0,0.0,0.0,180.666667,506.666667,180.442804,Up
2,content_000005d4ced12088,client_9958f0a7ae1df715,2025-05-01,2025-07-01,2025-08-01,2025-10-01,0.000000,0.0,216.666667,254.0,...,0.0,0.0,0.0,0.0,0.0,0.0,216.666667,487.666667,125.076923,Up
3,content_000005d4ced12088,client_9958f0a7ae1df715,2025-06-01,2025-08-01,2025-09-01,2025-11-01,0.333333,1.0,378.333333,742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,378.333333,280.333333,-25.903084,Down
4,content_000005d4ced12088,client_9958f0a7ae1df715,2025-07-01,2025-09-01,2025-10-01,2025-12-01,0.666667,1.0,506.666667,524.0,...,0.0,0.0,0.0,0.0,0.0,0.0,506.666667,167.000000,-67.039474,Down



BLOCK 4 COMPLETE


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.